# 🚀 AnimeKu Studio — 100% Free Video Ingestion Engine (Google Colab)
Notebook ini digunakan untuk mengunduh episode anime dari Nyaa.si, mengekstrak Subtitle Indonesia resmi, mengonversi Multi-Resolusi HLS `fMP4 Single-File` (1080p, 720p, 480p) dengan GPU gratis, dan mengunggahnya langsung ke **Cloudflare R2** serta mendaftarkannya ke database **Cloudflare D1** tanpa memakan kuota internet atau komputasi pribadi Anda.

> **Tips GPU Gratis:** Buka menu **Runtime** -> **Change runtime type** -> Pilih **T4 GPU** untuk akselerasi rendering hardware super cepat dari Google!

In [ ]:
#@title 📦 1. Persiapan Environment & Instalasi Tools (1-Click Run) { display-mode: "form" }
import os, subprocess

print("⏳ [1/4] Menginstal aria2c, rclone, Node.js 20, dan tools dasar...")
!apt-get update -qq > /dev/null
!apt-get install -y -qq aria2 rclone curl ca-certificates xz-utils > /dev/null

# Setup Node.js 20 jika belum ada
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get install -y -qq nodejs > /dev/null

print("⏳ [2/4] Memeriksa & Mengaktifkan Akselerasi GPU NVIDIA NVENC...")
has_gpu = subprocess.run("nvidia-smi", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
has_nvenc = subprocess.run("ffmpeg -encoders 2>/dev/null | grep -q 'h264_nvenc'", shell=True).returncode == 0

if has_gpu and not has_nvenc:
    print("🚀 [GPU DETECTED] Memasang build resmi BtbN FFmpeg NVENC Hardware Transcoder...")
    !wget -q -O /tmp/ffmpeg.tar.xz https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/ffmpeg-master-latest-linux64-gpl.tar.xz
    !tar -xf /tmp/ffmpeg.tar.xz -C /tmp --strip-components=1
    !sudo cp /tmp/bin/ffmpeg /usr/local/bin/ffmpeg
    !sudo cp /tmp/bin/ffprobe /usr/local/bin/ffprobe
    !sudo chmod +x /usr/local/bin/ffmpeg /usr/local/bin/ffprobe
    !rm -rf /tmp/ffmpeg.tar.xz /tmp/bin /tmp/doc
    print("✅ FFmpeg NVENC GPU Hardware Transcoder Aktif!")
elif not has_gpu:
    print("ℹ️ Tidak ada GPU terdeteksi, menggunakan CPU fallback.")
    !apt-get install -y -qq ffmpeg > /dev/null
else:
    print("✅ FFmpeg NVENC sudah siap digunakan!")

print("⏳ [3/4] Menyiapkan repositori runner AnimeKu Worker (Public)...")
if not os.path.exists("/content/animeku-worker"):
    !git clone --depth 1 https://github.com/MrHyzee/animeku-worker.git /content/animeku-worker
else:
    %cd /content/animeku-worker
    !git pull

os.chdir("/content/animeku-worker")

print("⏳ [4/4] Menginstal dependencies Node.js...")
!npm install --no-audit --no-fund > /dev/null 2>&1

print("\n🎉 Setup Selesai! Environment Google Colab siap mengeksekusi pipeline dengan GPU T4.")

In [ ]:
#@title 🔑 2. Konfigurasi Kredensial Cloudflare (Cukup Sekali) { display-mode: "form" }
#@markdown > **Catatan Arsitektur:** Parameter judul anime, nomor episode, dan magnet **100% diatur secara dinamis dari Admin Panel**! Server Colab ini akan otomatis memproses anime apapun yang Anda antrekan dari Admin Panel.
import os

try:
    from google.colab import userdata
    CF_API_TOKEN = userdata.get("CF_API_TOKEN")
    R2_ACCESS_KEY = userdata.get("R2_ACCESS_KEY")
    R2_SECRET_KEY = userdata.get("R2_SECRET_KEY")
except Exception:
    CF_API_TOKEN = None
    R2_ACCESS_KEY = None
    R2_SECRET_KEY = None

CF_ACCOUNT_ID = "5b5d0b730a79c86cf659a0d01f86ba9f" #@param {type:"string"}
CF_DATABASE_ID = "b51144f3-8050-4349-815c-ecb3402510f7" #@param {type:"string"}
CF_API_TOKEN = CF_API_TOKEN or "" #@param {type:"string"}
R2_ACCESS_KEY = R2_ACCESS_KEY or "" #@param {type:"string"}
R2_SECRET_KEY = R2_SECRET_KEY or "" #@param {type:"string"}
R2_BUCKET_NAME = "animekuapp-media" #@param {type:"string"}
DEFAULT_MODE = "direct" #@param ["direct", "hls"]
STORAGE_DRIVER = "r2" #@param ["r2", "hetzner"]

os.environ["CF_ACCOUNT_ID"] = CF_ACCOUNT_ID
os.environ["CF_DATABASE_ID"] = CF_DATABASE_ID
os.environ["CF_API_TOKEN"] = CF_API_TOKEN
os.environ["R2_ACCESS_KEY"] = R2_ACCESS_KEY
os.environ["R2_SECRET_KEY"] = R2_SECRET_KEY
os.environ["R2_BUCKET_NAME"] = R2_BUCKET_NAME
os.environ["STORAGE_DRIVER"] = STORAGE_DRIVER
os.environ["DEFAULT_MODE"] = DEFAULT_MODE
MODE = DEFAULT_MODE

if STORAGE_DRIVER == "r2" and (not R2_ACCESS_KEY or not R2_SECRET_KEY):
    print("⚠️ PERHATIAN: R2_ACCESS_KEY atau R2_SECRET_KEY masih kosong di form Cell 2!")
    print("👉 Harap isi form R2_ACCESS_KEY dan R2_SECRET_KEY di atas agar file bisa diunggah ke Cloudflare R2.")
else:
    print("✅ Kredensial Cloudflare R2 & D1 Berhasil Diaktifkan!")
    print("👉 Siap mendengarkan antrean job dari Admin Panel.")

In [ ]:
#@title 🚀 3. Jalankan Always-On Worker Daemon (Otomatis Mendengarkan Admin Panel) { display-mode: "form" }
#@markdown Klik tombol ▶️ di bawah. Server Colab akan standby 24/7 dan otomatis memproses anime apapun yang Anda dispatch dari Admin Panel.
import os, subprocess, sys
from IPython.display import Javascript, display

MODE = os.environ.get("DEFAULT_MODE") or globals().get("MODE", "direct")

# Ambil kredensial dari Colab Secrets jika os.environ belum terisi
try:
    from google.colab import userdata
    if not os.environ.get("CF_API_TOKEN") and userdata.get("CF_API_TOKEN"):
        os.environ["CF_API_TOKEN"] = userdata.get("CF_API_TOKEN")
    if not os.environ.get("R2_ACCESS_KEY") and userdata.get("R2_ACCESS_KEY"):
        os.environ["R2_ACCESS_KEY"] = userdata.get("R2_ACCESS_KEY")
    if not os.environ.get("R2_SECRET_KEY") and userdata.get("R2_SECRET_KEY"):
        os.environ["R2_SECRET_KEY"] = userdata.get("R2_SECRET_KEY")
    if not os.environ.get("CF_ACCOUNT_ID"):
        os.environ["CF_ACCOUNT_ID"] = "5b5d0b730a79c86cf659a0d01f86ba9f"
    if not os.environ.get("CF_DATABASE_ID"):
        os.environ["CF_DATABASE_ID"] = "b51144f3-8050-4349-815c-ecb3402510f7"
    if not os.environ.get("R2_BUCKET_NAME"):
        os.environ["R2_BUCKET_NAME"] = "animekuapp-media"
except Exception:
    pass

# 1. Anti-Disconnect Heartbeat: Mensimulasikan heartbeat DOM agar Colab tidak tidur/idle
display(Javascript("""
  function ColabHeartbeat() {
    console.log("[AnimeKu Daemon] Heartbeat Keep-Alive: " + new Date().toLocaleTimeString());
    var btn = document.querySelector("colab-connect-button");
    if (btn) btn.click();
  }
  if (!window.animekuHeartbeatInterval) {
    window.animekuHeartbeatInterval = setInterval(ColabHeartbeat, 60000);
    console.log("🛡️ Anti-Sleep Heartbeat Aktif (Interval 60s)!");
  }
"""))

print("🟢 [ALWAYS-ON DAEMON] Server Colab Aktif Standby Mendengarkan Admin Panel!")
print("   👉 Anda bisa minimize tab ini atau buka HP.")
print("   👉 Kapanpun Anda membuat job di Admin Panel (anime apapun: Solo Leveling, One Piece, dsb.),")
print("   👉 Server ini akan otomatis langsung mengambil data job tersebut & memprosesnya dengan GPU T4!\n")

# Pastikan selalu berada di direktori runner animeku-worker
if os.path.exists("/content/animeku-worker"):
    os.chdir("/content/animeku-worker")

print("🔄 Memperbarui script runner ke commit terbaru dari GitHub...")
subprocess.run("git fetch origin main --quiet && git reset --hard origin/main --quiet", shell=True)

cmd = "node scripts/ingest.js worker-daemon --mode direct --worker google-colab"
print(f"⚡ Menjalankan Daemon: {cmd}\n")

# 2. Live stream output seketika ke terminal Colab (Zero-Buffering)
proc = subprocess.Popen(cmd, shell=True)
proc.wait()

if proc.returncode == 0:
    print("\n👋 Daemon dihentikan secara normal.")
else:
    print(f"\n❌ Proses terhenti dengan error code {proc.returncode}.")